# 00 — Setup & Configuration

This notebook configures the connection to your **Microsoft Foundry** project and verifies
connectivity. Run this once before running the other notebooks.

Configuration is saved to `workshop_config.json` so the investigation and deployment
notebooks can load it automatically.

### Prerequisites
- Python 3.10+
- Azure CLI installed and logged in (`az login`)
- A Microsoft Foundry project with a deployed model (e.g., `gpt-4o`)

In [ ]:
# Install dependencies
%pip install -q agent-framework-core agent-framework-foundry agent-framework-foundry-hosting agent-framework-devui azure-ai-projects azure-identity pydantic python-dotenv

In [1]:
# Foundry Connection
#
# Edit the 4 variables below to connect to your Foundry resource.
# If you've already run this notebook, the saved config will be loaded automatically.

import json
from pathlib import Path

# ──────────────────────────────────────────────────────────────
# ✏️  Edit these to connect to your Foundry resource
# ──────────────────────────────────────────────────────────────
RESOURCE_GROUP = "rg-foundry-iac-ops"         # e.g. "rg-my-foundry"
ACCOUNT_NAME = "fndryiac2ttkx3"           # e.g. "my-foundry-account"
PROJECT_NAME = "iac-ops-demo"           # e.g. "my-project"
MODEL_DEPLOYMENT_NAME = "gpt-4.1-mini"  # e.g. "gpt-4o"
# ──────────────────────────────────────────────────────────────

# Load saved config if variables are empty
config_path = Path("workshop_config.json")
if config_path.exists() and not ACCOUNT_NAME:
    with open(config_path) as f:
        config = json.load(f)
    RESOURCE_GROUP = config.get("RESOURCE_GROUP", RESOURCE_GROUP)
    ACCOUNT_NAME = config.get("ACCOUNT_NAME", ACCOUNT_NAME)
    PROJECT_NAME = config.get("PROJECT_NAME", PROJECT_NAME)
    MODEL_DEPLOYMENT_NAME = config.get("MODEL_DEPLOYMENT_NAME", MODEL_DEPLOYMENT_NAME)
    print(f"✅ Loaded saved config from {config_path}")

if not ACCOUNT_NAME:
    raise ValueError(
        "No Foundry connection configured. "
        "Edit the 4 variables above (RESOURCE_GROUP, ACCOUNT_NAME, PROJECT_NAME, MODEL_DEPLOYMENT_NAME)."
    )

PROJECT_ENDPOINT = f"https://{ACCOUNT_NAME}.services.ai.azure.com/api/projects/{PROJECT_NAME}"

print(f"Resource Group:      {RESOURCE_GROUP}")
print(f"Account Name:        {ACCOUNT_NAME}")
print(f"Project Name:        {PROJECT_NAME}")
print(f"Model Deployment:    {MODEL_DEPLOYMENT_NAME}")
print(f"Project Endpoint:    {PROJECT_ENDPOINT}")

Resource Group:      rg-foundry-iac-ops
Account Name:        fndryiac2ttkx3
Project Name:        iac-ops-demo
Model Deployment:    gpt-4.1-mini
Project Endpoint:    https://fndryiac2ttkx3.services.ai.azure.com/api/projects/iac-ops-demo


In [3]:
# Verify Foundry connectivity

from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

credential = AzureCliCredential()

af_client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL_DEPLOYMENT_NAME,
    credential=credential,
)

_test_agent = Agent(client=af_client, instructions="Reply with only the word 'OK'.", name="ConnTest")
_test_resp = await _test_agent.run("ping")
print(f"✅ Foundry connection verified: {_test_resp.text[:50]}")
del _test_agent, _test_resp

✅ Foundry connection verified: OK


In [4]:
# Save config for other notebooks

config = {
    "RESOURCE_GROUP": RESOURCE_GROUP,
    "ACCOUNT_NAME": ACCOUNT_NAME,
    "PROJECT_NAME": PROJECT_NAME,
    "MODEL_DEPLOYMENT_NAME": MODEL_DEPLOYMENT_NAME,
    "PROJECT_ENDPOINT": PROJECT_ENDPOINT,
}

with open("workshop_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("✅ Config saved to workshop_config.json")
print("   Run 01-investigation.ipynb next.")

✅ Config saved to workshop_config.json
   Run 01-investigation.ipynb next.
